In [24]:
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm.auto import tqdm
from sklearn.cluster import KMeans

#sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "geo_dataset"
TRAIN_DIR = DATA_DIR / "train"
LABELS_PATH = DATA_DIR / "train_labels.csv"

MODEL_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "country_gated_cells"
)

best_model_path = MODEL_DIR / "best_model.pt"
checkpoint_path = MODEL_DIR / "training_checkpoint.pt"

ABLATION_OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "gating_temperature_ablation"
)

ABLATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

results_path = (
    ABLATION_OUTPUT_DIR
    / "temperature_results.csv"
)

print("Model:", best_model_path)
print("Results:", results_path)

Model: /home/utn/poli22wo/Desktop/UTN/Semester 2/Deep Learning/Final Project/outputs/country_gated_cells/best_model.pt
Results: /home/utn/poli22wo/Desktop/UTN/Semester 2/Deep Learning/Final Project/outputs/gating_temperature_ablation/temperature_results.csv


In [4]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

MODEL_NAME = checkpoint["model_name"]
NUMBER_OF_COUNTRIES = checkpoint[
    "number_of_countries"
]
NUMBER_OF_CELLS = checkpoint[
    "number_of_cells"
]

country_to_index = checkpoint[
    "country_to_index"
]

index_to_country = checkpoint[
    "index_to_country"
]

cell_centres_tensor = checkpoint[
    "cell_centres"
].float()

cell_to_country_tensor = checkpoint[
    "cell_to_country"
].long()

print("Best epoch:", checkpoint["best_epoch"])
print("Best median:", checkpoint["best_median"])
print("Countries:", NUMBER_OF_COUNTRIES)
print("Cells:", NUMBER_OF_CELLS)

Best epoch: 24
Best median: 282.95383
Countries: 12
Cells: 96


In [5]:
df = pd.read_csv(LABELS_PATH)

df["country_index"] = df["country"].map(
    country_to_index
)

_, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["country"],
)

val_df = val_df.copy()

print("Validation images:", len(val_df))
display(val_df.head())

Validation images: 2352


,filename,country,iso,lat,lng,country_index
10836,8bce910418fc4a3dbeaee73d2cafc814.jpg,United_Kingdom,GB,50.377206,-4.464514,11
9048,c19f7011acbc405994165695531c741f.jpg,Finland,FI,61.281270,28.806198,1
9950,0888345ae8664ac9b29d053585a8fa68.jpg,Norway,NO,64.743564,12.846071,6
367,efcddc47084b4e288f4268c27817bd6e.jpg,Belarus,BY,55.088826,30.615250,0
6288,21215fa75acd48469e6fae1f7e25c5ed.jpg,Spain,ES,42.549467,-8.864157,8


In [7]:
processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME
)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=(
        NUMBER_OF_COUNTRIES
        + NUMBER_OF_CELLS
    ),
    ignore_mismatched_sizes=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

best_weights = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_weights)
model = model.to(device)
model.eval()

print("Device:", device)
print("Loaded best model.")

[transformers] You passed `num_labels=108` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 37541.35it/s]
[transformers] MobileViTV2ForImageClassification LOAD REPORT from: apple/mobilevitv2-1.0-imagenet1k-256
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([108])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([108, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Device: cuda
Loaded best model.


Image Processor and Dataset

In [9]:
class GeolocationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        processor,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        coordinates = torch.tensor(
            [
                row["lat"] / 90,
                row["lng"] / 180,
            ],
            dtype=torch.float32,
        )

        country_index = torch.tensor(
            row["country_index"],
            dtype=torch.long,
        )

        return (
            pixel_values,
            coordinates,
            country_index,
        )

In [10]:
val_dataset = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

In [11]:
BATCH_SIZE = 32

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [12]:
def haversine_km(lat1, lng1, lat2, lng2):
    radius = 6371.0088

    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    difference = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lng2 - lng1) / 2) ** 2
    )

    return (
        2
        * radius
        * np.arcsin(
            np.sqrt(np.clip(difference, 0, 1))
        )
    )

In [13]:
all_country_logits = []
all_cell_logits = []
all_coordinates = []
all_country_labels = []

model.eval()

with torch.inference_mode():

    for (
        images,
        coordinates,
        country_labels,
    ) in tqdm(
        val_loader,
        desc="Collecting validation logits",
    ):

        images = images.to(device)

        outputs = model(
            pixel_values=images
        ).logits

        country_logits = outputs[
            :, :NUMBER_OF_COUNTRIES
        ]

        cell_logits = outputs[
            :, NUMBER_OF_COUNTRIES:
        ]

        all_country_logits.append(
            country_logits.cpu()
        )

        all_cell_logits.append(
            cell_logits.cpu()
        )

        all_coordinates.append(
            coordinates
        )

        all_country_labels.append(
            country_labels
        )

In [14]:
all_country_logits = torch.cat(
    all_country_logits,
    dim=0,
)

all_cell_logits = torch.cat(
    all_cell_logits,
    dim=0,
)

all_coordinates = torch.cat(
    all_coordinates,
    dim=0,
)

all_country_labels = torch.cat(
    all_country_labels,
    dim=0,
)

print(
    "Country logits:",
    all_country_logits.shape,
)

print(
    "Cell logits:",
    all_cell_logits.shape,
)

print(
    "Coordinates:",
    all_coordinates.shape,
)

print(
    "Country labels:",
    all_country_labels.shape,
)

Country logits: torch.Size([2352, 12])
Cell logits: torch.Size([2352, 96])
Coordinates: torch.Size([2352, 2])
Country labels: torch.Size([2352])


Evaluation function

In [15]:
def evaluate_configuration(
    name,
    country_temperature=1.0,
    cell_temperature=1.0,
    use_country_gating=True,
):

    country_probabilities = torch.softmax(
        all_country_logits
        / country_temperature,
        dim=1,
    )

    cell_probabilities = torch.softmax(
        all_cell_logits
        / cell_temperature,
        dim=1,
    )

    if use_country_gating:

        country_weights_for_cells = (
            country_probabilities[
                :, cell_to_country_tensor
            ]
        )

        final_cell_probabilities = (
            cell_probabilities
            * country_weights_for_cells
        )

        final_cell_probabilities = (
            final_cell_probabilities
            / final_cell_probabilities.sum(
                dim=1,
                keepdim=True,
            ).clamp_min(1e-8)
        )

    else:

        final_cell_probabilities = (
            cell_probabilities
        )

    predicted_coordinates = (
        final_cell_probabilities
        @ cell_centres_tensor
    )

    predictions_degrees = (
        predicted_coordinates.numpy().copy()
    )

    coordinates_degrees = (
        all_coordinates.numpy().copy()
    )

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    return {
        "configuration": name,
        "country_temperature": (
            country_temperature
        ),
        "cell_temperature": (
            cell_temperature
        ),
        "country_gating": (
            use_country_gating
        ),
        "mean_km": np.mean(distances),
        "median_km": np.median(distances),
        "within_200": np.mean(
            distances < 200
        ),
        "within_750": np.mean(
            distances < 750
        ),
    }

Gating vs No Gating

In [16]:
baseline_results = [
    evaluate_configuration(
        name="No country gating",
        country_temperature=1.0,
        cell_temperature=1.0,
        use_country_gating=False,
    ),

    evaluate_configuration(
        name="Standard country gating",
        country_temperature=1.0,
        cell_temperature=1.0,
        use_country_gating=True,
    ),
]

baseline_results_df = pd.DataFrame(
    baseline_results
)

display(baseline_results_df)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,No country gating,1.0,1.0,False,568.781433,305.868439,0.379252,0.737245
1,Standard country gating,1.0,1.0,True,581.492859,282.953979,0.400085,0.737670


In [17]:
predicted_countries = (
    all_country_logits.argmax(dim=1)
)

country_accuracy = (
    predicted_countries
    == all_country_labels
).float().mean().item()

print(
    f"Country accuracy: "
    f"{country_accuracy:.2%}"
)

Country accuracy: 67.05%


In [18]:
temperature_values = [
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    2.00,
]

temperature_results = []

In [19]:
for country_temperature in temperature_values:

    for cell_temperature in temperature_values:

        result = evaluate_configuration(
            name="Country gating",
            country_temperature=(
                country_temperature
            ),
            cell_temperature=(
                cell_temperature
            ),
            use_country_gating=True,
        )

        temperature_results.append(result)

In [20]:
for cell_temperature in temperature_values:

    result = evaluate_configuration(
        name="No country gating",
        country_temperature=1.0,
        cell_temperature=(
            cell_temperature
        ),
        use_country_gating=False,
    )

    temperature_results.append(result)

In [21]:
temperature_results_df = pd.DataFrame(
    temperature_results
)

temperature_results_df = (
    temperature_results_df
    .sort_values(
        by=[
            "median_km",
            "mean_km",
        ]
    )
    .reset_index(drop=True)
)

display(
    temperature_results_df.head(15)
)

temperature_results_df.to_csv(
    results_path,
    index=False,
)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,Country gating,0.25,1.00,True,594.954468,270.527466,0.402636,0.736820
1,Country gating,0.25,1.25,True,595.381775,271.900574,0.395833,0.741071
2,Country gating,0.50,1.25,True,589.353760,273.172974,0.394133,0.740646
3,Country gating,0.50,1.00,True,589.304443,273.369263,0.399660,0.735969
4,Country gating,0.25,0.75,True,596.299622,275.322876,0.410714,0.734694
5,Country gating,0.50,0.75,True,591.192078,275.460449,0.409864,0.735969
6,Country gating,0.75,1.00,True,584.775269,276.478668,0.399660,0.738946
7,Country gating,0.25,1.50,True,597.070068,277.206207,0.390306,0.740221
8,Country gating,0.75,0.75,True,587.384277,278.343628,0.409014,0.737245
9,Country gating,0.50,1.50,True,590.691223,279.419861,0.387755,0.741071


In [22]:
best_gated_result = (
    temperature_results_df[
        temperature_results_df[
            "country_gating"
        ]
    ]
    .iloc[0]
)

best_ungated_result = (
    temperature_results_df[
        ~temperature_results_df[
            "country_gating"
        ]
    ]
    .iloc[0]
)

display(
    pd.DataFrame([
        best_gated_result,
        best_ungated_result,
    ])
)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,Country gating,0.25,1.0,True,594.954468,270.527466,0.402636,0.736820
40,No country gating,1.00,0.5,False,591.858337,291.950623,0.401361,0.725765


In [23]:
BEST_COUNTRY_TEMPERATURE = 0.25
BEST_CELL_TEMPERATURE = 1.00

best_configuration = (
    temperature_results_df[
        (
            temperature_results_df[
                "country_temperature"
            ]
            == BEST_COUNTRY_TEMPERATURE
        )
        &
        (
            temperature_results_df[
                "cell_temperature"
            ]
            == BEST_CELL_TEMPERATURE
        )
        &
        (
            temperature_results_df[
                "country_gating"
            ]
        )
    ]
    .copy()
)

best_configuration_path = (
    ABLATION_OUTPUT_DIR
    / "best_configuration.csv"
)

best_configuration.to_csv(
    best_configuration_path,
    index=False,
)

display(best_configuration)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,Country gating,0.25,1.0,True,594.954468,270.527466,0.402636,0.73682
